In [6]:
!pip install pandas

  Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [7]:
import pandas as pd

In [34]:
income_df = pd.read_csv('tract_income.csv')

In [35]:
income_df = pd.read_csv('tract_income.csv', skiprows=[1])
income_df = income_df[['GEO_ID', 'B19013_001E']].rename(columns={
    'B19013_001E': 'median_income'
})

In [36]:
#extract numeric tract ID from GEO_ID
income_df['GEOID'] = income_df['GEO_ID'].str.split('US').str[-1]

In [37]:
#drop non-numerica/missing median income value rows
income_df = income_df[pd.to_numeric(income_df['median_income'], errors='coerce').notna()]
income_df['median_income'] = income_df['median_income'].astype(float)


In [38]:
crosswalk_df = pd.read_csv('tract_nta_crosswalk.csv')
crosswalk_df['GEOID'] = crosswalk_df['GEOID'].astype(str)

In [39]:
#merge on GEOID to get NTA for each tract's median income
merged_df = pd.merge(income_df, crosswalk_df[['GEOID', 'NTACode']], on='GEOID', how='inner')

In [40]:
#aggregate to get average median income by NTA
nta_income = merged_df.groupby('NTACode')['median_income'].median().reset_index()

In [41]:
#apply minmax normalization to median income values
lowest = nta_income['median_income'].min()
highest = nta_income['median_income'].max()
nta_income['income_score'] = 1 - ((nta_income['median_income'] - lowest) / (highest - lowest))

In [42]:
print(nta_income.head(10))
print(f"\nTotal NTAs: {len(nta_income)}")

  NTACode  median_income  income_score
0  BK0101       133636.0      0.405188
1  BK0102       144083.0      0.345659
2  BK0103        44085.0      0.915467
3  BK0104        88457.0      0.662627
4  BK0201       181645.0      0.131623
5  BK0202       155905.5      0.278292
6  BK0203       129817.0      0.426949
7  BK0204       139329.0      0.372748
8  BK0301        93003.0      0.636723
9  BK0302        71786.0      0.757621

Total NTAs: 200


In [ ]:
#62 missing NTAs are non-residential (parks, cemeteries, airports)
#identified by NTACode suffix 71, 91, 92, 93. These are expected to have no census data
crosswalk_all_ntas = crosswalk_df['NTACode'].unique()
missing = set(crosswalk_all_ntas) - set(nta_income['NTACode'])
print(sorted(missing))

['BK0261', 'BK0471', 'BK0571', 'BK0771', 'BK0891', 'BK1091', 'BK1391', 'BK1771', 'BK1891', 'BK1892', 'BK1893', 'BK5591', 'BK5691', 'BK5692', 'BK5693', 'BX0291', 'BX0391', 'BX0491', 'BX0492', 'BX0991', 'BX1071', 'BX1091', 'BX1161', 'BX1271', 'BX2691', 'BX2791', 'BX2891', 'MN0191', 'MN0661', 'MN1191', 'MN1291', 'MN1292', 'MN6491', 'QN0151', 'QN0171', 'QN0191', 'QN0261', 'QN0271', 'QN0571', 'QN0572', 'QN0573', 'QN0574', 'QN0761', 'QN0791', 'QN0871', 'QN0891', 'QN1091', 'QN1191', 'QN1371', 'QN1491', 'QN8081', 'QN8191', 'QN8291', 'QN8381', 'QN8491', 'QN8492', 'SI0191', 'SI0291', 'SI0391', 'SI9591', 'SI9592', 'SI9593']


In [44]:
nta_income[['NTACode', 'median_income', 'income_score']].to_csv('income_by_nta.csv', index=False)